# M8-1: the ablation, on the A100

Trains and scores M8-1's three studies (README, *The ablation (M8-1)*) from #64's merge (`aec24a1`), on Colab's G4 runtime, an `NVIDIA A100-SXM4-80GB`. About 57 h of compute (450–510 units) over several sessions. Every session is the same: **Runtime → Run all**. Each cell picks up where the last session stopped.

**Before connecting, every session:**
- Runtime → Change runtime type → **G4** → Save.
- Key icon on the left: `WANDB_API_KEY` has **notebook access** switched on.

## **Step 1: At the start of every session**

#### Cell A: the GPU, Drive, wandb and the dataset

In [ ]:
import hashlib, os, pathlib, subprocess

# The GPU first, before anything else runs: every ablation run trains and is scored on the sweep's GPU (PROJECT_PLAN §3.3)
GPU = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True,
                     capture_output=True, text=True).stdout.strip()
print(GPU or "no GPU")
assert GPU == "NVIDIA A100-SXM4-80GB", (
    f"this runtime has {GPU or 'no GPU'}: Runtime > Disconnect and delete runtime, then try G4 again later")

from google.colab import drive, userdata
drive.mount("/content/drive")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

COMMIT = "aec24a145f5d8a566d6d1e7b5b26775c262ff95c"   # #64's merge: every ablation session trains from it
DRIVE = pathlib.Path("/content/drive/MyDrive/msml612")
OUT = DRIVE / "ablation"                             # the ablation's runs
SWEEP = DRIVE / "sweep"                              # the sweep's runs: read only, nothing is written to it
DATASET = DRIVE / "physionet_set_a.npz"
assert (SWEEP / "ext1+2_random_seed42" / "summary.json").exists(), f"no sweep runs in {SWEEP}"
assert hashlib.sha256(DATASET.read_bytes()).hexdigest() == \
    "ce6ec696eee3440a08ff6cd81b311b2a4050487c396dfed66e93edc7d7458070", "not the dataset the sweep trained on"
print("Drive, wandb and the dataset: ready")

It prints `NVIDIA A100-SXM4-80GB`, then `Drive, wandb and the dataset: ready`.

If it stops at the GPU check, nothing else has run. Choose Runtime → **Disconnect and delete runtime** right away so it stops using units. Never use another GPU, not even the 40 GB A100.

#### Cell B: the code, pinned to the ablation's commit

In [ ]:
REPO = pathlib.Path("/content/msml612_project")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/jkelle11-source/msml612_project.git", str(REPO)],
                   check=True)
os.chdir(REPO)
subprocess.run(["git", "checkout", "-q", COMMIT], check=True)
HEAD = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True).stdout.strip()
assert HEAD == COMMIT, f"checked out {HEAD}, not the ablation's commit"
!git log --oneline -1
!uv sync --locked
TORCH = subprocess.run(["uv", "run", "python", "-c",
                        "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"],
                       capture_output=True, text=True).stdout.strip()
print(TORCH)
assert TORCH == "2.14.1+cu130 13.0 True", "not the locked environment, or torch cannot see the GPU"

Check two lines:
- **The commit:** `aec24a1 Merge pull request #64 from jkelle11-source/m8-1_sankha`, in every session, even after later merges into `main`.
- **The torch line:** `2.14.1+cu130 13.0 True`.

## **Step 2: The rerun check, once, about 2 h**

This retrains the sweep's `ext1+2_random_seed42` from the ablation's commit, into a folder of its own, and checks that it reproduces the sweep's numbers exactly. The studies reuse the sweep's runs as their 64 / 50 / 2 cells only if it does.

#### Cell C

In [ ]:
!uv run python -m diffusion.sweep --tag rerun-check --regime random --variants ext1+2 --seeds 42 --out {OUT}/rerun-check --dataset {DATASET}
check = subprocess.run(["uv", "run", "python", "-m", "diffusion.sweep", "--compare", str(SWEEP), "--regime", "random",
                        "--variants", "ext1+2", "--seeds", "42", "--out", str(OUT / "rerun-check")],
                       capture_output=True, text=True)
print(check.stdout + check.stderr)
check.check_returncode()    # stops Run all here if it does not reproduce: the studies wait

- **The first lines:** `1 cells on cuda -> /content/drive/MyDrive/msml612/ablation/rerun-check`, then `[1/1] ext1+2_random_seed42`.
- **It passes** when the last line printed is `ext1+2_random_seed42: the same numbers`.
- **If it stops** with `differs in ...` and `not every cell reproduces`, don't go on. Save the printed lines for the team, because the sweep's values would then have to be trained again in each study (README, *The ablation*).
- **In a later session** this cell finds the run already finished and compares again in a few seconds.

## **Step 3: The three studies, about 55 h**

Each command trains a study's other two values at seeds 42–44, into `ablation/<study>_<value>`. `d_model` goes first because M8-2's MAE-vs-latency plot needs it.

Run again, each command picks up where it stopped. Finished runs are skipped, and a lost run resumes from its last checkpoint, losing at most 50 epochs (about 9 minutes).

#### Cell D: `d_model` 128 / 256, 6 runs, about 15 h

In [ ]:
!uv run python -m diffusion.ablation --study d_model --tag ablation --out {OUT} --dataset {DATASET}

- **The first line:** `d_model: diffusion.channels = [128, 256] on ext1+2 random at seeds 42 43 44, on cuda -> /content/drive/MyDrive/msml612/ablation`.
- **Early check:** wandb shows `ext1+2_random_seed42` tagged `ablation` and `d_model_128`.
- **The 15 h is rough**, because `d_model` was never timed on the A100. Note how long the first run's epochs take.

#### Cell E: mask-encoder layers 1 / 3, 12 runs, about 23 h

In [ ]:
!uv run python -m diffusion.ablation --study encoder_layers --tag ablation --out {OUT} --dataset {DATASET}

- **The first line:** `encoder_layers: model.mask_encoder_layers = [1, 3] on ext1+2 random; ext1 block-aware at seeds 42 43 44, on cuda -> /content/drive/MyDrive/msml612/ablation`.
- Each run is scored on both holdouts.

#### Cell F: diffusion steps T = 100 / 200, 6 runs, about 16 h

In [ ]:
!uv run python -m diffusion.ablation --study steps --tag ablation --out {OUT} --dataset {DATASET}

- **The first line:** `steps: diffusion.num_steps = [100, 200] on ext1+2 random at seeds 42 43 44, on cuda -> /content/drive/MyDrive/msml612/ablation`.
- The T = 200 runs are the slowest: their validation and scoring take 4× as many denoising steps.

## **Ending a session yourself**

Stop the running cell (■), then run this cell so Google Drive finishes uploading before you disconnect. A session that is lost can't do this, and the driver redoes what it lost. Run all also reaches this cell on its own after the last study finishes.

#### Cell G

In [ ]:
drive.flush_and_unmount()

## **When the ablation is done**

- Download this notebook from the open tab (File → Download → .ipynb), not from Drive, as `notebooks/M8-1_Ablation_<i>.ipynb`, one per stretch. Strip its Colab identity fields before it goes in Git, as for M7-1.
- The table (`diffusion.ablation_table`) is not in this notebook. It runs on a CPU runtime from `main` (README, *The table*).